# Candidate Model Architecture

## Objective
Define and justify candidate model architectures for predicting term deposit subscription (`y`) in the Bank Marketing project.

This notebook focuses on model selection rationale, evaluation strategy, and a practical shortlist for implementation.

## Problem Context
- Binary classification on tabular data (subscribe: yes/no)
- Class imbalance is present
- Mix of categorical and numeric features
- Business need: improve conversion efficiency and reduce unnecessary calls
- Existing pipeline already handles train/val/test split and preprocessing

## Candidate Model Architectures

### 1) Regularized Logistic Regression
**Why use it:**
- Strong and interpretable baseline for binary classification
- Fast to train, easy to debug, and provides probabilities for threshold tuning
- Works well with one-hot encoded categorical features and scaled numeric features

**Project fit:**
- Establishes a trustworthy benchmark before moving to more complex models

**Source:**
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

### 2) Elastic Net Logistic Regression
**Why use it:**
- Handles sparse and correlated features better than pure L2 in many settings
- Performs implicit feature selection through L1 component

**Project fit:**
- Useful when one-hot expansion and engineered indicators create correlated feature groups

**Source:**
- https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

### 3) Random Forest (Class-Weighted)
**Why use it:**
- Captures nonlinearities and interactions without heavy manual specification
- Robust baseline ensemble with stable out-of-the-box behavior

**Project fit:**
- Serves as a tree-based benchmark against boosting methods

**Source:**
- https://link.springer.com/article/10.1023/A:1010933404324

### 4) XGBoost (Gradient Boosted Trees)
**Why use it:**
- Excellent tabular performance in many structured-data problems
- Handles nonlinear effects and high-order interactions effectively
- Supports class imbalance handling (`scale_pos_weight`)

**Project fit:**
- Likely strong candidate given mixed features and engineered interaction signals

**Source:**
- https://dl.acm.org/doi/10.1145/2939672.2939785

### 5) LightGBM
**Why use it:**
- Fast, memory efficient, and competitive on tabular tasks
- Suitable for quick iteration and hyperparameter sweeps

**Project fit:**
- Good alternative to XGBoost when runtime and iteration speed are priorities

**Source:**
- https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html

### 6) CatBoost
**Why use it:**
- Strong performance on categorical-heavy tabular data
- Ordered boosting design helps reduce overfitting in categorical contexts

**Project fit:**
- Dataset has multiple categorical attributes (`job`, `marital`, `education`, `contact`, `poutcome`)

**Source:**
- https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html

### 7) Balanced Ensemble Methods
**Why use it:**
- Designed for imbalanced settings by balancing subsets during training
- Can improve minority-class recall while preserving ensemble robustness

**Project fit:**
- Useful when campaign objective emphasizes finding likely subscribers (positive class)

**Source:**
- https://imbalanced-learn.org/stable/references/ensemble.html

### 8) Transformer-Based Tabular Models (Advanced)
**Why use it:**
- Can model complex cross-feature interactions
- Potentially competitive with tree methods in some tabular settings

**Project fit:**
- Best treated as advanced/experimental after strong boosted-tree baselines are established

**Source:**
- https://arxiv.org/abs/2012.06678

## Recommended Priority Order for This Project
1. Regularized Logistic Regression (baseline)
2. XGBoost
3. LightGBM and/or CatBoost
4. Random Forest
5. Balanced Ensemble variant
6. Transformer-based tabular model (optional advanced benchmark)

## Evaluation Strategy
Because the target is imbalanced, use multiple metrics and select thresholds by business objective:
- PR-AUC (primary)
- Recall, Precision, F1
- ROC-AUC (secondary)
- Threshold tuning (e.g., maximize recall subject to minimum precision)
- Optional: probability calibration checks (Brier score, reliability curve)

## Runnable Model Comparison Setup
The cells below run a practical benchmark across candidate architectures using your preprocessed artifacts.

Expected input files from `02_preprocessing.ipynb`:
- `data/bank-full-X-model.parquet`
- `data/bank-full-y-model.parquet`

In [24]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

DATA_DIR = Path("data")
X_PATH = DATA_DIR / "bank-full-X-model.parquet"
y_PATH = DATA_DIR / "bank-full-y-model.parquet"

if not X_PATH.exists() or not y_PATH.exists():
    raise FileNotFoundError(
        "Expected preprocessed files were not found. Run 02_preprocessing.ipynb first."
    )

X_model = pd.read_parquet(X_PATH)
y_model = pd.read_parquet(y_PATH)

# Convert target to a clean 1D numeric array if needed.
if isinstance(y_model, pd.DataFrame):
    if y_model.shape[1] != 1:
        raise ValueError("Target parquet should contain exactly one target column.")
    y_model = y_model.iloc[:, 0]

if y_model.dtype == "object":
    y_map = {"no": 0, "yes": 1}
    if set(y_model.unique()).issubset(set(y_map.keys())):
        y_model = y_model.map(y_map)

y_model = pd.Series(y_model).astype(int)

categorical_cols = X_model.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X_model.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)

print(f"X shape: {X_model.shape}")
print(f"y shape: {y_model.shape}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")
print("Class distribution:")
print(y_model.value_counts(normalize=True).rename("proportion").sort_index())

X shape: (45211, 21)
y shape: (45211,)
Categorical columns: 8
Numeric columns: 13
Class distribution:
target
0    0.883015
1    0.116985
Name: proportion, dtype: float64


In [25]:
# Stratified split: 64% train, 16% val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_model,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (28934, 21)
Val shape: (7234, 21)
Test shape: (9043, 21)


In [26]:
def evaluate_model(model_name, estimator, X_fit, y_fit, X_eval, y_eval, threshold=0.5):
    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )
    model.fit(X_fit, y_fit)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        raw_score = model.decision_function(X_eval)
        y_score = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min() + 1e-12)
    else:
        raise ValueError(f"{model_name} has no probability-like output method.")

    y_pred = (y_score >= threshold).astype(int)

    return {
        "model": model_name,
        "pipeline": model,
        "roc_auc": roc_auc_score(y_eval, y_score),
        "pr_auc": average_precision_score(y_eval, y_score),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall": recall_score(y_eval, y_pred, zero_division=0),
        "f1": f1_score(y_eval, y_pred, zero_division=0),
    }


def build_candidate_estimators(pos_weight):
    estimators = {
        "logistic_l2_balanced": LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "logistic_elasticnet": LogisticRegression(
            penalty="elasticnet",
            l1_ratio=0.5,
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "random_forest_balanced": RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }

    # Optional gradient boosting libraries. If unavailable, those models are skipped.
    try:
        from xgboost import XGBClassifier

        estimators["xgboost"] = XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            scale_pos_weight=pos_weight,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        )
    except Exception as e:
        print(f"Skipping XGBoost: {e}")

    try:
        from lightgbm import LGBMClassifier

        estimators["lightgbm"] = LGBMClassifier(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    except Exception as e:
        print(f"Skipping LightGBM: {e}")

    try:
        from catboost import CatBoostClassifier

        estimators["catboost"] = CatBoostClassifier(
            iterations=700,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
        )
    except Exception as e:
        print(f"Skipping CatBoost: {e}")

    return estimators

In [27]:
# Build candidates and evaluate on validation split
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
pos_weight = n_neg / max(n_pos, 1)

estimators = build_candidate_estimators(pos_weight=pos_weight)
results = []
fitted_pipelines = {}

for name, estimator in estimators.items():
    row = evaluate_model(
        model_name=name,
        estimator=estimator,
        X_fit=X_train,
        y_fit=y_train,
        X_eval=X_val,
        y_eval=y_val,
        threshold=0.50,
    )
    fitted_pipelines[name] = row.pop("pipeline")
    results.append(row)

val_results = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)
val_results

[LightGBM] [Info] Number of positive: 3385, number of negative: 25549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000960 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 191
[LightGBM] [Info] Number of data points in the train set: 28934, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/Users/kofibadu/miniforge3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,roc_auc,pr_auc,precision,recall,f1
0,xgboost,0.800779,0.460957,0.351333,0.622931,0.449275
1,catboost,0.804051,0.458619,0.354126,0.644208,0.457023
2,lightgbm,0.805550,0.453917,0.364014,0.621749,0.459188
3,random_forest_balanced,0.782244,0.418603,0.459459,0.442080,0.450602
4,logistic_elasticnet,0.772185,0.410645,0.275013,0.633570,0.383542
5,logistic_l2_balanced,0.772150,0.410456,0.274590,0.633570,0.383131


In [28]:
# Fit best validation model on train+val and report test metrics
best_model_name = val_results.loc[0, "model"]
best_estimator = estimators[best_model_name]

test_row = evaluate_model(
    model_name=f"{best_model_name}_test",
    estimator=best_estimator,
    X_fit=X_train_val,
    y_fit=y_train_val,
    X_eval=X_test,
    y_eval=y_test,
    threshold=0.50,
)

test_row.pop("pipeline")
pd.DataFrame([test_row])

,model,roc_auc,pr_auc,precision,recall,f1
0,xgboost_test,0.79756,0.458593,0.355591,0.634216,0.455688


## Threshold Tuning (Business-Aligned Selection)
Use validation probabilities to select thresholds based on campaign goals.

Example objective used below:
- maximize recall while enforcing minimum precision (default: `precision >= 0.35`)

In [29]:
from sklearn.metrics import precision_recall_curve


def fit_pipeline(estimator, X_fit, y_fit):
    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )
    model.fit(X_fit, y_fit)
    return model


def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    raise ValueError("Model does not support probability-like scoring.")


def select_threshold_by_precision(y_true, y_score, min_precision=0.35):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)

    # precision_recall_curve returns len(thresholds) = len(precisions)-1
    candidate_rows = []
    for i, t in enumerate(thresholds):
        p = precisions[i + 1]
        r = recalls[i + 1]
        if p >= min_precision:
            candidate_rows.append((float(t), float(p), float(r)))

    if not candidate_rows:
        # Fallback: choose threshold with best F1 on validation
        best = None
        for i, t in enumerate(thresholds):
            p = precisions[i + 1]
            r = recalls[i + 1]
            f1 = 0.0 if (p + r) == 0 else 2 * p * r / (p + r)
            row = (float(t), float(p), float(r), float(f1))
            if best is None or row[3] > best[3]:
                best = row
        return {
            "threshold": best[0],
            "precision": best[1],
            "recall": best[2],
            "selection_rule": "fallback_best_f1",
        }

    # Choose highest recall among precision-qualified thresholds
    candidate_rows.sort(key=lambda x: (x[2], x[1]), reverse=True)
    t, p, r = candidate_rows[0]
    return {
        "threshold": t,
        "precision": p,
        "recall": r,
        "selection_rule": f"max_recall_with_precision_at_least_{min_precision}",
    }

In [30]:
# Tune thresholds on top validation models and evaluate on test
TOP_K = 3
MIN_PRECISION = 0.35

top_models = val_results.head(TOP_K)["model"].tolist()
threshold_rows = []

for model_name in top_models:
    estimator = estimators[model_name]
    pipeline = fit_pipeline(estimator, X_train, y_train)

    val_score = get_scores(pipeline, X_val)
    selected = select_threshold_by_precision(
        y_true=y_val,
        y_score=val_score,
        min_precision=MIN_PRECISION,
    )

    # Refit on train+val before final test evaluation
    final_pipeline = fit_pipeline(estimator, X_train_val, y_train_val)
    test_score = get_scores(final_pipeline, X_test)
    test_pred = (test_score >= selected["threshold"]).astype(int)

    threshold_rows.append(
        {
            "model": model_name,
            "selected_threshold": selected["threshold"],
            "val_precision_at_threshold": selected["precision"],
            "val_recall_at_threshold": selected["recall"],
            "selection_rule": selected["selection_rule"],
            "test_pr_auc": average_precision_score(y_test, test_score),
            "test_roc_auc": roc_auc_score(y_test, test_score),
            "test_precision": precision_score(y_test, test_pred, zero_division=0),
            "test_recall": recall_score(y_test, test_pred, zero_division=0),
            "test_f1": f1_score(y_test, test_pred, zero_division=0),
        }
    )

threshold_results = pd.DataFrame(threshold_rows).sort_values(
    ["test_recall", "test_precision"], ascending=False
).reset_index(drop=True)

threshold_results

[LightGBM] [Info] Number of positive: 3385, number of negative: 25549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000958 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 191
[LightGBM] [Info] Number of data points in the train set: 28934, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/Users/kofibadu/miniforge3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 4231, number of negative: 31937
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 191
[LightGBM] [Info] Number of data points in the train set: 36168, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/kofibadu/miniforge3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,selected_threshold,val_precision_at_threshold,val_recall_at_threshold,selection_rule,test_pr_auc,test_roc_auc,test_precision,test_recall,test_f1
0,catboost,0.495330,0.350608,0.647754,max_recall_with_precision_at_least_0.35,0.457604,0.800530,0.350285,0.637996,0.452261
1,lightgbm,0.488875,0.353175,0.631206,max_recall_with_precision_at_least_0.35,0.458672,0.800762,0.347646,0.635161,0.449348
2,xgboost,0.502737,0.353454,0.622931,max_recall_with_precision_at_least_0.35,0.458593,0.797560,0.359828,0.633270,0.458904


## Interpretation of Results (Model by Model)

Use this section to explain the benchmark table in plain business terms.

### 1) XGBoost
**Observed pattern (validation):** highest PR-AUC, strong ROC-AUC, good recall with moderate precision.

**Interpretation:**
-  is the strongest overall ranking model for identifying likely subscribers under class imbalance.
- Higher PR-AUC means it is better at prioritizing positive-class customers near the top of the score distribution.
- Moderate precision indicates some false positives remain, but recall is strong enough to capture many likely subscribers.

**When to use:**
- Best default choice when your objective is overall targeting quality and high campaign yield.

### 2) CatBoost
**Observed pattern (validation):** very close to XGBoost in PR-AUC, slightly stronger recall than XGBoost at default threshold.

**Interpretation:**
- CatBoost is highly competitive and may be preferable if your campaign values finding as many potential subscribers as possible.
- Strong categorical handling can be useful for this dataset's feature profile.
- If threshold tuning is aggressive for recall, CatBoost may become the practical winner in some settings.

**When to use:**
- Good alternative if recall is prioritized over precision and you want robust performance on categorical-heavy data.

### 3) LightGBM
**Observed pattern (validation):** PR-AUC slightly below XGBoost/CatBoost, best ROC-AUC among top models, balanced precision-recall profile.

**Interpretation:**
- LightGBM ranks instances well globally (high ROC-AUC), but its positive-class prioritization (PR-AUC) is a bit weaker than XGBoost.
- It often provides a good precision-recall balance and trains quickly.

**When to use:**
- Strong choice for fast iteration and tuning, especially when runtime is important.

### 4) Random Forest (Balanced)
**Observed pattern (validation):** lower PR-AUC than boosting models, highest precision among tested models at default threshold, lower recall.

**Interpretation:**
- Random Forest is conservative: it predicts fewer positives, so precision improves but many true subscribers are missed.
- This can be useful when call budget is tight and false positives are expensive.

**When to use:**
- Useful for precision-focused campaigns where the team wants fewer, higher-confidence calls.

### 5) Logistic Regression (Elastic Net)
**Observed pattern (validation):** lower PR-AUC/ROC-AUC than tree ensembles, high recall but low precision at default threshold.

**Interpretation:**
- The model captures broad directional signal but underfits complex nonlinear interactions compared with boosted trees.
- High recall + low precision means many predicted positives are not actual subscribers.

**When to use:**
- Good interpretable baseline and benchmark for explainability; not the top production performer here.

### 6) Logistic Regression (L2 Balanced)
**Observed pattern (validation):** nearly identical to Elastic Net and below tree ensembles.

**Interpretation:**
- Similar tradeoff profile: strong baseline transparency, weaker ranking performance versus boosted trees.
- Confirms that linear decision boundaries are likely insufficient for full predictive power in this feature space.

**When to use:**
- Baseline model for governance, explainability, and sanity checks.

---

## Interpretation of Threshold-Tuned Results (`threshold_results`)
When you run threshold tuning, interpret outcomes as follows:

- Higher **test_recall** means you capture more likely subscribers (fewer missed opportunities).
- Higher **test_precision** means fewer wasted calls (better call efficiency).
- Higher **test_f1** indicates a more balanced precision-recall tradeoff.
- **test_pr_auc** remains threshold-independent and reflects overall ranking quality.

Practical decision rule:
- If campaign budget is constrained, choose the row with higher precision at acceptable recall.
- If growth targets dominate, choose the row with highest recall while meeting minimum precision.
- Keep the selected threshold tied to business constraints (agent capacity, call cost, conversion value).